# AE pre-sweep (original architecture)

Original-architecture autoencoder (`base_channels=32`, `depth=3`) applied
across the d-sweep. **Part A** trains on all 804 ages (in-sample);
**Part B** trains on the block-stride 80/10/10 split, early-stops on val,
reports on test. Each Part runs a POD baseline, an AE d-sweep, a focus-d
eval at d=16, and a Layer-2 deep dive on PCA(d=16 to 2) of the d=16 latent.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Anchor to the repo root so paleoreco imports and relative file paths
# resolve regardless of where Jupyter was launched from.
REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from sklearn.decomposition import PCA

from paleoreco.data import (
    PaleoFieldDataset,
    apply_zscore,
    build_prior_cube,
    compute_zscore_stats,
)
from paleoreco.splits import (
    assign_event_label,
    block_stride_split,
    summarize_split,
)
from paleoreco.models.autoencoder import ConvAE, count_parameters
from paleoreco.train_ae import set_seed, train
from paleoreco.eval.ae import plot_loss_curves, reconstruct_split
from paleoreco.eval.shared import (
    compute_pod_time_coefficients,
    compute_split_metrics,
    partition_latent_2d,
    per_cell_rmse_celsius,
    per_mode_learning_accuracy,
    plot_latent_2d,
    plot_latent_sweep,
    plot_per_cell_rmse,
    plot_per_cluster_pod_distributions,
    plot_per_cluster_reconstructions,
    plot_per_mode_learning_curves,
    plot_recon_distribution,
    plot_reconstructions,
    pod_fit,
    pod_predict,
)

plt.rcParams["figure.dpi"] = 110

In [ ]:
SEED = 42
LATENT_DIMS = (2, 4, 8, 16, 32, 64)
D_FOCUS = 16
MAX_EPOCHS = 250
PATIENCE = 25
BATCH_SIZE = 32

# Layer-2 viz: toggle PCA on/off and its target dim. PCA off requires
# D_FOCUS == 2 (direct-latent viz, Bousquet-style replicate).
LAYER2_USE_PCA = True
LAYER2_PCA_DIM = 2
LAYER2_TAG = f"d{D_FOCUS}" + (f"_pca{LAYER2_PCA_DIM}" if LAYER2_USE_PCA else "")

# Original (v1-default) architecture and optimiser settings.
BASE_STAR = 32
DEPTH_STAR = 3
LR_STAR = 1e-3
WD_STAR = 1e-4

OUT_DIR = Path(REPO_ROOT) / "outputs"
FIG_DIR = OUT_DIR / "figures" / "02_ae_pre_sweep"
CSV_DIR = OUT_DIR / "csvs" / "02_ae_pre_sweep"
PARTA_CSV = CSV_DIR / "partA.csv"
PARTB_CSV = CSV_DIR / "partB.csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)


def _snapshot_cb_factory(snapshots: list[dict]):
    """Return an epoch-callback that appends a CPU clone of the model state.

    Used at d=D_FOCUS to capture per-epoch state_dicts for the per-POD-mode
    learning curves; the closure keeps the snapshot list as a write target.
    """
    def cb(_epoch_idx, model):
        snapshots.append(
            {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        )
    return cb

In [ ]:
data = build_prior_cube(
    prior_csv=os.path.join(REPO_ROOT, "data/Prior.csv"),
    cache_path=os.path.join(REPO_ROOT, "data/cache/prior_cube.npz"),
)
cube = data["cube"]
ages = data["ages"]
lats = data["lats"]
lons = data["lons"]
valid = data["valid"]
N_AGES = len(ages)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cube: {cube.shape}  ages: [{ages.min()}, {ages.max()}] yr BP")
print(f"device: {device}")

## Part A: 100% train (in-sample)

In [ ]:
all_idx_A = np.arange(N_AGES)
stats_A = compute_zscore_stats(cube, train_age_indices=all_idx_A, valid=valid)
cube_z_A = apply_zscore(cube, stats_A)
mask_A = stats_A["safe_valid"]
print(f"Part A safe_valid cells: {int(mask_A.sum())} / {mask_A.size}")

dataset_A = PaleoFieldDataset(cube_z_A, mask_A, all_idx_A)
loader_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=True)
print(f"Part A: {len(dataset_A)} samples, {len(loader_A)} batches")

### POD baseline (all-ages fit)

In [ ]:
max_k_A = max(LATENT_DIMS)
pod_basis_A = pod_fit(cube_z_A, all_idx_A, mask_A, max_k=max_k_A, random_state=SEED)

partA_pod_rows = []
for k in LATENT_DIMS:
    pod_pred = pod_predict(cube_z_A, all_idx_A, pod_basis_A, k=k)
    m = compute_split_metrics(cube_z_A[all_idx_A], pod_pred, mask_A)
    partA_pod_rows.append({
        "latent_dim":    k,
        "pod_mse_z":     m["mse_z"],
        "pod_rmse_z":    m["rmse_z"],
        "pod_rrmse":     m["rrmse"],
        "pod_r_squared": m["r_squared"],
        "pod_E_d":       m["E_d"],
    })

print(pd.DataFrame(partA_pod_rows).to_string(index=False))

### AE d-sweep (original arch on all-ages)

In [ ]:
partA_ae_rows = []
partA_focus = {}
partA_focus_snapshots: list[dict] = []  # populated only at d=D_FOCUS

for k in LATENT_DIMS:
    print(f"\n[Part A {k:>2}d] training...")
    set_seed(SEED)
    model = ConvAE(latent_dim=k, base_channels=BASE_STAR, depth=DEPTH_STAR)
    n_params = count_parameters(model)

    # Capture per-epoch state_dicts only at d=D_FOCUS, used downstream
    # for the per-POD-mode learning curves.
    epoch_callback = _snapshot_cb_factory(partA_focus_snapshots) if k == D_FOCUS else None

    t0 = time.perf_counter()
    out = train(
        model, loader_A, val_loader=None,
        mask=mask_A, zscore_std=stats_A["std"],
        lr=LR_STAR, weight_decay=WD_STAR,
        max_epochs=MAX_EPOCHS, patience=None,
        device=device, seed=SEED,
        verbose=False, progress=True,
        epoch_callback=epoch_callback,
    )
    seconds = time.perf_counter() - t0

    model.load_state_dict(out["best_state_dict"])
    truth_z, pred_z = reconstruct_split(model, dataset_A, device=device, batch_size=BATCH_SIZE)
    m = compute_split_metrics(truth_z, pred_z, mask_A)
    partA_ae_rows.append({
        "latent_dim":     k,
        "ae_mse_z":       m["mse_z"],
        "ae_rmse_z":      m["rmse_z"],
        "ae_rrmse":       m["rrmse"],
        "ae_r_squared":   m["r_squared"],
        "ae_E_d":         m["E_d"],
        "n_params":       n_params,
        "epochs_trained": out["epochs_trained"],
    })

    # Persist combined df incrementally so an interrupt keeps prior rows.
    partA_df = pd.DataFrame(partA_ae_rows).merge(
        pd.DataFrame(partA_pod_rows), on="latent_dim", how="left",
    )
    partA_df.to_csv(PARTA_CSV, index=False)

    print(f"  ae_E_d={m['E_d']:.3f}  ae_R²={m['r_squared']:.3f}  "
          f"epochs={out['epochs_trained']}  ({seconds:.0f}s)")

    if k == D_FOCUS:
        partA_focus = {
            "history":  out["history"],
            "truth_z":  truth_z,
            "pred_z":   pred_z,
            "model":    model,
        }
        print(f"  captured {len(partA_focus_snapshots)} per-epoch snapshots")

In [ ]:
partA_df = pd.DataFrame(partA_ae_rows).merge(
    pd.DataFrame(partA_pod_rows), on="latent_dim", how="left",
)
partA_df.to_csv(PARTA_CSV, index=False)
print(partA_df.to_string(index=False))

In [ ]:
fig = plot_latent_sweep(
    latent_dims=LATENT_DIMS,
    model_E_d=[r["ae_E_d"] for r in partA_ae_rows],
    pod_E_d=[r["pod_E_d"] for r in partA_pod_rows],
    model_label="AE (original arch)",
    save_path=str(FIG_DIR / "partA_d_sweep_ae_vs_pod.png"),
)
plt.show()

### Focus-d eval at d=16

In [ ]:
fig = plot_loss_curves(
    partA_focus["history"],
    save_path=str(FIG_DIR / f"partA_loss_curves_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_reconstructions(
    partA_focus["truth_z"], partA_focus["pred_z"],
    stats_A, ages, lats, lons,
    save_path=str(FIG_DIR / f"partA_reconstructions_d{D_FOCUS}.png"),
)
plt.show()

rmse_c_A = per_cell_rmse_celsius(
    partA_focus["truth_z"], partA_focus["pred_z"], stats_A, mask_A,
)
fig = plot_per_cell_rmse(
    rmse_c_A, lats, lons,
    save_path=str(FIG_DIR / f"partA_per_cell_rmse_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_recon_distribution(
    partA_focus["truth_z"], partA_focus["pred_z"], stats_A, mask_A,
    save_path=str(FIG_DIR / f"partA_recon_distribution_d{D_FOCUS}.png"),
)
plt.show()

### Layer-2 deep dive

In [ ]:
# Encode every age through the d=D_FOCUS AE; the Layer-2 viz is
# descriptive, so coordinates are computed on all 804 ages.
partA_focus["model"].eval()
with torch.no_grad():
    encode_loader_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
    Z_A_chunks = []
    for batch in encode_loader_A:
        batch = batch.to(device, non_blocking=True)
        Z_A_chunks.append(partA_focus["model"].encode(batch).cpu().numpy())
Z_A = np.concatenate(Z_A_chunks, axis=0)
print(f"Part A d={D_FOCUS} latents: {Z_A.shape}")

if LAYER2_USE_PCA:
    pca_A = PCA(n_components=LAYER2_PCA_DIM, random_state=SEED)
    Z2_A = pca_A.fit_transform(Z_A)
    print(f"PCA({LAYER2_PCA_DIM}) explained variance ratios: {pca_A.explained_variance_ratio_}")
else:
    if Z_A.shape[1] != 2:
        raise ValueError(
            f"LAYER2_USE_PCA=False requires D_FOCUS=2; got latent dim {Z_A.shape[1]}"
        )
    Z2_A = Z_A

labels_sign_A = partition_latent_2d(Z2_A, "z2_sign")
labels_km4_A = partition_latent_2d(Z2_A, 4, random_state=SEED)
print(f"sign cluster sizes: {[int((labels_sign_A == c).sum()) for c in np.unique(labels_sign_A)]}")
print(f"km4 cluster sizes:  {[int((labels_km4_A == c).sum()) for c in np.unique(labels_km4_A)]}")

In [ ]:
event_labels = assign_event_label(ages)

fig = plot_latent_2d(
    Z2_A, event_labels, color_label="D-O event index (0 = between)",
    title=f"Part A ({LAYER2_TAG}): latent coloured by D-O event",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_scatter_by_event.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_A, ages, color_label="age (yr BP)",
    title=f"Part A ({LAYER2_TAG}): latent coloured by age",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_scatter_by_age.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_A, labels_sign_A, color_label="cluster (sign of axis 2)",
    title=f"Part A ({LAYER2_TAG}): partition by sign of axis 2",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_partition_sign.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_A, labels_km4_A, color_label="cluster (KMeans-4)",
    title=f"Part A ({LAYER2_TAG}): KMeans(4) partition",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_partition_km4.png"),
)
plt.show()

In [ ]:
pod_a_k_A = compute_pod_time_coefficients(cube_z_A, mask_A, pod_basis_A)
TOP_K = list(range(5))

fig = plot_per_cluster_pod_distributions(
    pod_a_k_A, labels_sign_A, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_pod_sign.png"),
)
plt.show()

fig = plot_per_cluster_pod_distributions(
    pod_a_k_A, labels_km4_A, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_per_cluster_pod_km4.png"),
)
plt.show()

In [ ]:
# anomaly = x - mean = z * std (zero-mean by construction).
cube_anomaly_A = cube_z_A * stats_A["std"][None]

fig = plot_per_cluster_reconstructions(
    cube_anomaly_A, labels_sign_A, lats, lons, mask_A,
    var_names=("mtco", "mtwa"), unit_label="°C anomaly",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_cluster_physical_sign.png"),
)
plt.show()

fig = plot_per_cluster_reconstructions(
    cube_anomaly_A, labels_km4_A, lats, lons, mask_A,
    var_names=("mtco", "mtwa"), unit_label="°C anomaly",
    save_path=str(FIG_DIR / f"partA_{LAYER2_TAG}_cluster_physical_km4.png"),
)
plt.show()

In [ ]:
# Per-POD-mode learning curves: replay each per-epoch snapshot of the
# d=16 model on all 804 ages, project the reconstructions through
# pod_basis_A, and compare to truth POD time-coefficients to get e_k(epoch)
# per Bousquet Eq. 23.
replay_model_A = ConvAE(latent_dim=D_FOCUS, base_channels=BASE_STAR, depth=DEPTH_STAR).to(device)
replay_model_A.eval()

n_ep_A = len(partA_focus_snapshots)
ae_a_k_per_epoch_A = np.empty((n_ep_A, N_AGES, max_k_A), dtype=np.float64)
replay_loader_A = DataLoader(dataset_A, batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for ep, sd in enumerate(partA_focus_snapshots):
        replay_model_A.load_state_dict(sd)
        recons = []
        for batch in replay_loader_A:
            batch = batch.to(device, non_blocking=True)
            x_hat, _ = replay_model_A(batch)
            recons.append(x_hat.cpu().numpy())
        recon = np.concatenate(recons, axis=0)
        ae_a_k_per_epoch_A[ep] = compute_pod_time_coefficients(recon, mask_A, pod_basis_A)

e_k_A = per_mode_learning_accuracy(pod_a_k_A, ae_a_k_per_epoch_A)
fig = plot_per_mode_learning_curves(
    e_k_A, ks_to_show=list(range(min(10, max_k_A))),
    save_path=str(FIG_DIR / f"partA_per_mode_learning_d{D_FOCUS}.png"),
)
plt.show()

print("Part A final-epoch e_k (top 10 POD modes):")
for k in range(min(10, max_k_A)):
    print(f"  k={k:>2d}: e_k = {e_k_A[-1, k]:.3f}")

## Part B: 80/10/10 split (test reporting)

In [ ]:
split = block_stride_split(N_AGES)
print(summarize_split(ages, split))

stats_B = compute_zscore_stats(cube, train_age_indices=split["train"], valid=valid)
cube_z_B = apply_zscore(cube, stats_B)
mask_B = stats_B["safe_valid"]
print(f"\nPart B safe_valid cells: {int(mask_B.sum())} / {mask_B.size}")

train_ds_B = PaleoFieldDataset(cube_z_B, mask_B, split["train"])
val_ds_B   = PaleoFieldDataset(cube_z_B, mask_B, split["val"])
test_ds_B  = PaleoFieldDataset(cube_z_B, mask_B, split["test"])
train_loader_B = DataLoader(train_ds_B, batch_size=BATCH_SIZE, shuffle=True)
val_loader_B   = DataLoader(val_ds_B,   batch_size=BATCH_SIZE, shuffle=False)
print(f"Part B sizes: train={len(train_ds_B)}  val={len(val_ds_B)}  test={len(test_ds_B)}")

### POD baseline (train-fit, test eval)

In [ ]:
max_k_B = max(LATENT_DIMS)
pod_basis_B = pod_fit(cube_z_B, split["train"], mask_B, max_k=max_k_B, random_state=SEED)

partB_pod_rows = []
for k in LATENT_DIMS:
    pod_pred_test = pod_predict(cube_z_B, split["test"], pod_basis_B, k=k)
    m = compute_split_metrics(cube_z_B[split["test"]], pod_pred_test, mask_B)
    partB_pod_rows.append({
        "latent_dim":    k,
        "pod_mse_z":     m["mse_z"],
        "pod_rmse_z":    m["rmse_z"],
        "pod_rrmse":     m["rrmse"],
        "pod_r_squared": m["r_squared"],
        "pod_E_d":       m["E_d"],
    })

print(pd.DataFrame(partB_pod_rows).to_string(index=False))

### AE d-sweep (original arch, early-stop on val, test report)

In [ ]:
partB_ae_rows = []
partB_focus = {}
partB_focus_snapshots: list[dict] = []  # populated only at d=D_FOCUS

for k in LATENT_DIMS:
    print(f"\n[Part B {k:>2}d] training...")
    set_seed(SEED)
    model = ConvAE(latent_dim=k, base_channels=BASE_STAR, depth=DEPTH_STAR)
    n_params = count_parameters(model)

    epoch_callback = _snapshot_cb_factory(partB_focus_snapshots) if k == D_FOCUS else None

    t0 = time.perf_counter()
    out = train(
        model, train_loader_B, val_loader=val_loader_B,
        mask=mask_B, zscore_std=stats_B["std"],
        lr=LR_STAR, weight_decay=WD_STAR,
        max_epochs=MAX_EPOCHS, patience=PATIENCE,
        device=device, seed=SEED,
        verbose=False, progress=True,
        epoch_callback=epoch_callback,
    )
    seconds = time.perf_counter() - t0

    model.load_state_dict(out["best_state_dict"])
    truth_z, pred_z = reconstruct_split(model, test_ds_B, device=device, batch_size=BATCH_SIZE)
    m = compute_split_metrics(truth_z, pred_z, mask_B)
    partB_ae_rows.append({
        "latent_dim":     k,
        "ae_mse_z":       m["mse_z"],
        "ae_rmse_z":      m["rmse_z"],
        "ae_rrmse":       m["rrmse"],
        "ae_r_squared":   m["r_squared"],
        "ae_E_d":         m["E_d"],
        "n_params":       n_params,
        "epochs_trained": out["epochs_trained"],
        "best_epoch":     out["best_epoch"],
    })

    partB_df = pd.DataFrame(partB_ae_rows).merge(
        pd.DataFrame(partB_pod_rows), on="latent_dim", how="left",
    )
    partB_df.to_csv(PARTB_CSV, index=False)

    print(f"  ae_E_d={m['E_d']:.3f}  ae_R²={m['r_squared']:.3f}  "
          f"epochs={out['epochs_trained']}  best={out['best_epoch']}  "
          f"({seconds:.0f}s)")

    if k == D_FOCUS:
        partB_focus = {
            "history":     out["history"],
            "truth_z":     truth_z,
            "pred_z":      pred_z,
            "model":       model,
            "best_epoch":  out["best_epoch"],
        }
        print(f"  captured {len(partB_focus_snapshots)} per-epoch snapshots")

In [ ]:
partB_df = pd.DataFrame(partB_ae_rows).merge(
    pd.DataFrame(partB_pod_rows), on="latent_dim", how="left",
)
partB_df.to_csv(PARTB_CSV, index=False)
print(partB_df.to_string(index=False))

In [ ]:
fig = plot_latent_sweep(
    latent_dims=LATENT_DIMS,
    model_E_d=[r["ae_E_d"] for r in partB_ae_rows],
    pod_E_d=[r["pod_E_d"] for r in partB_pod_rows],
    model_label="AE (original arch, test)",
    save_path=str(FIG_DIR / "partB_d_sweep_ae_vs_pod.png"),
)
plt.show()

### Focus-d eval at d=16 (test)

In [ ]:
fig = plot_loss_curves(
    partB_focus["history"], best_epoch=partB_focus["best_epoch"],
    save_path=str(FIG_DIR / f"partB_loss_curves_d{D_FOCUS}.png"),
)
plt.show()

test_size = partB_focus["truth_z"].shape[0]
sample_indices = np.linspace(0, test_size - 1, 5).astype(int).tolist()
fig = plot_reconstructions(
    partB_focus["truth_z"], partB_focus["pred_z"],
    stats_B, ages[split["test"]], lats, lons,
    sample_indices=sample_indices,
    save_path=str(FIG_DIR / f"partB_reconstructions_d{D_FOCUS}.png"),
)
plt.show()

rmse_c_B = per_cell_rmse_celsius(
    partB_focus["truth_z"], partB_focus["pred_z"], stats_B, mask_B,
)
fig = plot_per_cell_rmse(
    rmse_c_B, lats, lons,
    save_path=str(FIG_DIR / f"partB_per_cell_rmse_d{D_FOCUS}.png"),
)
plt.show()

fig = plot_recon_distribution(
    partB_focus["truth_z"], partB_focus["pred_z"], stats_B, mask_B,
    save_path=str(FIG_DIR / f"partB_recon_distribution_d{D_FOCUS}.png"),
)
plt.show()

### Layer-2 deep dive

In [ ]:
# Encode all 804 ages through the Part B d=D_FOCUS model. cube_z_B
# uses the train-only stats the model was trained on; feeding all ages
# through is consistent with the model's input distribution.
partB_focus["model"].eval()
all_idx_B = np.arange(N_AGES)
all_ds_B = PaleoFieldDataset(cube_z_B, mask_B, all_idx_B)
with torch.no_grad():
    encode_loader_B = DataLoader(all_ds_B, batch_size=BATCH_SIZE, shuffle=False)
    Z_B_chunks = []
    for batch in encode_loader_B:
        batch = batch.to(device, non_blocking=True)
        Z_B_chunks.append(partB_focus["model"].encode(batch).cpu().numpy())
Z_B = np.concatenate(Z_B_chunks, axis=0)
print(f"Part B d={D_FOCUS} latents (all 804 ages): {Z_B.shape}")

if LAYER2_USE_PCA:
    pca_B = PCA(n_components=LAYER2_PCA_DIM, random_state=SEED)
    Z2_B = pca_B.fit_transform(Z_B)
    print(f"PCA({LAYER2_PCA_DIM}) explained variance ratios: {pca_B.explained_variance_ratio_}")
else:
    if Z_B.shape[1] != 2:
        raise ValueError(
            f"LAYER2_USE_PCA=False requires D_FOCUS=2; got latent dim {Z_B.shape[1]}"
        )
    Z2_B = Z_B

labels_sign_B = partition_latent_2d(Z2_B, "z2_sign")
labels_km4_B = partition_latent_2d(Z2_B, 4, random_state=SEED)
print(f"sign cluster sizes: {[int((labels_sign_B == c).sum()) for c in np.unique(labels_sign_B)]}")
print(f"km4 cluster sizes:  {[int((labels_km4_B == c).sum()) for c in np.unique(labels_km4_B)]}")

In [ ]:
fig = plot_latent_2d(
    Z2_B, event_labels, color_label="D-O event index (0 = between)",
    title=f"Part B ({LAYER2_TAG}): latent coloured by D-O event",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_scatter_by_event.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_B, ages, color_label="age (yr BP)",
    title=f"Part B ({LAYER2_TAG}): latent coloured by age",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_scatter_by_age.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_B, labels_sign_B, color_label="cluster (sign of axis 2)",
    title=f"Part B ({LAYER2_TAG}): partition by sign of axis 2",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_partition_sign.png"),
)
plt.show()

fig = plot_latent_2d(
    Z2_B, labels_km4_B, color_label="cluster (KMeans-4)",
    title=f"Part B ({LAYER2_TAG}): KMeans(4) partition",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_partition_km4.png"),
)
plt.show()

In [ ]:
pod_a_k_B = compute_pod_time_coefficients(cube_z_B, mask_B, pod_basis_B)

fig = plot_per_cluster_pod_distributions(
    pod_a_k_B, labels_sign_B, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_pod_sign.png"),
)
plt.show()

fig = plot_per_cluster_pod_distributions(
    pod_a_k_B, labels_km4_B, ks_to_show=TOP_K,
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_per_cluster_pod_km4.png"),
)
plt.show()

In [ ]:
cube_anomaly_B = cube_z_B * stats_B["std"][None]

fig = plot_per_cluster_reconstructions(
    cube_anomaly_B, labels_sign_B, lats, lons, mask_B,
    var_names=("mtco", "mtwa"), unit_label="°C anomaly",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_cluster_physical_sign.png"),
)
plt.show()

fig = plot_per_cluster_reconstructions(
    cube_anomaly_B, labels_km4_B, lats, lons, mask_B,
    var_names=("mtco", "mtwa"), unit_label="°C anomaly",
    save_path=str(FIG_DIR / f"partB_{LAYER2_TAG}_cluster_physical_km4.png"),
)
plt.show()

In [ ]:
# Per-POD-mode learning curves: same replay-and-project pattern as Part A,
# using pod_basis_B (train-fit) on all 804 ages of cube_z_B.
replay_model_B = ConvAE(latent_dim=D_FOCUS, base_channels=BASE_STAR, depth=DEPTH_STAR).to(device)
replay_model_B.eval()

n_ep_B = len(partB_focus_snapshots)
ae_a_k_per_epoch_B = np.empty((n_ep_B, N_AGES, max_k_B), dtype=np.float64)
replay_loader_B = DataLoader(all_ds_B, batch_size=BATCH_SIZE, shuffle=False)
with torch.no_grad():
    for ep, sd in enumerate(partB_focus_snapshots):
        replay_model_B.load_state_dict(sd)
        recons = []
        for batch in replay_loader_B:
            batch = batch.to(device, non_blocking=True)
            x_hat, _ = replay_model_B(batch)
            recons.append(x_hat.cpu().numpy())
        recon = np.concatenate(recons, axis=0)
        ae_a_k_per_epoch_B[ep] = compute_pod_time_coefficients(recon, mask_B, pod_basis_B)

e_k_B = per_mode_learning_accuracy(pod_a_k_B, ae_a_k_per_epoch_B)
fig = plot_per_mode_learning_curves(
    e_k_B, ks_to_show=list(range(min(10, max_k_B))),
    save_path=str(FIG_DIR / f"partB_per_mode_learning_d{D_FOCUS}.png"),
)
plt.show()

print("Part B final-epoch e_k (top 10 POD modes):")
for k in range(min(10, max_k_B)):
    print(f"  k={k:>2d}: e_k = {e_k_B[-1, k]:.3f}")

## Summary

Produces, in both Parts:
- POD baseline at every d in `LATENT_DIMS`.
- AE d-sweep at the winner architecture.
- Focus-d artefacts at d=16: loss curves, reconstruction grid, per-cell
  RMSE map, error distribution histogram.
- Layer-2 deep dive on PCA(d=16 -> 2) latents: scatters coloured by D-O
  event and by age; sign-of-PCA_2 and KMeans(4) partitions; per-cluster
  POD coefficient distributions and per-cluster physical-space composites
  in degC anomaly; Bousquet per-POD-mode learning curves e_k(epoch).